# ModClark Parameters Optmization

This notebook will contain process for optmizing ModClark IUH parameters $T_c$ and $R$. 
Parameter opmization here, follows the the methodology [described here](../../README.MD).

In [1]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import sys
from tqdm.notebook import tqdm

In [2]:
# Custom Modules
project_root_path = Path.cwd().parent.parent
sys.path.append(str(project_root_path / 'src'))

from hydrology.modclark_model import ModClarkModel
from utils.data_utils import glimpse

## Load and Analyze Storm Events Metada

In [3]:
# Load events metadata
events_metadata_path = project_root_path / 'data/gold/tabular/detected_storm_events.parquet'
storm_events_metadata = pd.read_parquet(events_metadata_path)

In [4]:
# Analyze storm events metadata
glimpse(storm_events_metadata)

Rows: 62258
Columns: 25
                          Null Count  Dtype                First Values
                          ----------  -----                -------------
event_start               0           datetime64[ns, UTC]  [2015-11-30T08:45:00.000000000, 2017-11-06T15:00:00.000000000, 2017-12-06T00:30:00.000000000, 2017-12-23T08:45:00.000000000, 2018-03-01T16:00:00.000000000]
event_end                 0           datetime64[ns, UTC]  [2015-12-03T23:45:00.000000000, 2017-11-10T21:45:00.000000000, 2017-12-09T16:00:00.000000000, 2017-12-25T15:30:00.000000000, 2018-03-03T13:15:00.000000000]
month                     0           int64                [11, 11, 12, 12, 3]
ppt_start                 0           datetime64[ns, UTC]  [2015-11-30T09:15:00.000000000, 2017-11-06T15:30:00.000000000, 2017-12-06T01:00:00.000000000, 2017-12-23T09:15:00.000000000, 2018-03-01T16:30:00.000000000]
ppt_end                   0           datetime64[ns, UTC]  [2015-12-02T20:15:00.000000000, 2017-11-08T12:45

In [6]:
print(f"\nTotal number of storm events: {len(storm_events_metadata)}")
storm_events_metadata['ws_id'] = storm_events_metadata['storm_id'].str.split('_').str[0]
unique_watersheds = storm_events_metadata['ws_id'].unique()
print(f"Number of unique watersheds: {len(unique_watersheds)}")
print(f"\nDistribution of storm events per watershed:")
print(storm_events_metadata.groupby('ws_id').size().reset_index(name='event_count')['event_count'].describe().round(0))
avg_events_per_watershed = storm_events_metadata.groupby('ws_id').size().mean()
print(f"\nNumber of watersheds with less than 10 storm events: {storm_events_metadata.groupby('ws_id').size()[storm_events_metadata.groupby('ws_id').size() < 10].count()}")
print(f"Number of watersheds with less than 5 storm events: {storm_events_metadata.groupby('ws_id').size()[storm_events_metadata.groupby('ws_id').size() < 5].count()}")
print(f"Average response time (in minutes): {storm_events_metadata['response_min'].mean():.2f}")
print(f"Average precipitation duration (in minutes): {storm_events_metadata['ppt_dutation_min'].mean():.2f}")
print(f"Number of storm events with response time > 60 minutes: {storm_events_metadata[storm_events_metadata['response_min'] > 60].shape[0]}")
print(f"Number of watershed with response time > 60 minutes: {storm_events_metadata[storm_events_metadata['response_min'] > 60]['ws_id'].nunique()}")
print(f"Number of storm watersheds with response time no more than 30 minutes: {storm_events_metadata[storm_events_metadata['response_min'] <= 30]['ws_id'].nunique()}")
print(f"Number of watersheds with more than 1 precipitation station: {storm_events_metadata[storm_events_metadata['n_ppt_stations'] > 1]['ws_id'].nunique()}")
print(f"Number of storm events with no precipitation after peakflow: {storm_events_metadata[storm_events_metadata['total_ppt_after_peak_mm'] == 0].shape[0]}")
print(f"Number of watersheds  with no precipitation after peakflow: {storm_events_metadata[storm_events_metadata['total_ppt_after_peak_mm'] == 0]['ws_id'].nunique()}")

print(f"""
Distribution of storm events per watershed that follows the criteria:
    - Number of storm events per watershed >= 5
    - Response time <= 30 minutes
    - Total precipitation after peakflow = 0 mm
      """)
filtered_storm_events_metada = storm_events_metadata[(storm_events_metadata['response_min'] <= 30) & 
                                             (storm_events_metadata['total_ppt_after_peak_mm'] == 0)]
filtered_storm_events_metada = filtered_storm_events_metada.groupby('ws_id').filter(lambda x: len(x) >= 5)
print(filtered_storm_events_metada.groupby('ws_id').size().reset_index(name='event_count')['event_count'].describe().round(0))

print(f"\nNumber of storm events that follow the criteria: {len(filtered_storm_events_metada)}")




Total number of storm events: 62258
Number of unique watersheds: 461

Distribution of storm events per watershed:
count    461.0
mean     135.0
std      107.0
min        2.0
25%       57.0
50%      109.0
75%      190.0
max      571.0
Name: event_count, dtype: float64

Number of watersheds with less than 10 storm events: 17
Number of watersheds with less than 5 storm events: 7
Average response time (in minutes): 69.18
Average precipitation duration (in minutes): 1923.81
Number of storm events with response time > 60 minutes: 608
Number of watershed with response time > 60 minutes: 191
Number of storm watersheds with response time no more than 30 minutes: 437
Number of watersheds with more than 1 precipitation station: 83
Number of storm events with no precipitation after peakflow: 18557
Number of watersheds  with no precipitation after peakflow: 456

Distribution of storm events per watershed that follows the criteria:
    - Number of storm events per watershed >= 5
    - Response time

In order to reduce computational demand, potentially "bad" storm events was dropped from the modeling, following the criteria:
1. No precipitation after peak flow (it guaratee more uniform hydrographs)
2. Storm events with response time less or equal to 30min. 30 min is a reasonable threshold for watershed to start the response. Response greater than 30min is probably due to data miss alighnment or spatial varibility not captured by precipitation stations. 
3. Finally, after filtering with these two criteria, watersheds with less than 5 storm events was also dropped from study. 

The number of total events to optmize dropped from **62.2k to 17.2k**.  
The number of watersheds dropped from **461 to 406**. 

## $R$ and $T_c$ Parameters Optmization

In [9]:
filtered_storm_events_metada.sample(1)

,event_start,event_end,month,ppt_start,ppt_end,total_precipitation_mm,total_ppt_after_peak_mm,response_min,ppt_dutation_min,event_duration_min,...,ppt_prior_7d,ppt_prior_3d,ppt_prior_24h,ppt_prior_6h,ppt_prior_3h,ppt_post_3h,ppt_post_6h,ppt_post_24h,storm_id,ws_id
56445,2016-04-28 01:15:00+00:00,2016-04-28 06:30:00+00:00,4,2016-04-28 01:45:00+00:00,2016-04-28 03:15:00+00:00,231.292138,0.0,30.0,105,330,...,0.130863,0.076083,0.066953,0.04565,0.04565,0.0,0.0,0.0,03337000_99,03337000


filter